# Phase 5 — Choosing the top level and building the hierarchy

The bubble UI needs a broad top level ("I want to help people") that drills down into fields and then jobs.

- **Numeric groups (Phase 3)** match career interests but split fields apart: nurses and nursing assistants
  ended up in different groups, and security / facilities managers landed in "Help & Heal".
- **Text clusters (Phase 2)** keep fields together but match interests poorly.

**Idea:** combine both. Each source is reduced with PCA, scaled to the same size, and weighted.
Code: `src/hybrid_groups.py`, `src/build_hierarchy.py`

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")
import json
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from hybrid_groups import hybrid_matrix, hybrid_groups, TEXT_WEIGHT, N_GROUPS
from load_onet import PROCESSED, ROOT

interests = pd.read_csv(PROCESSED / "interests.csv", index_col="code")["top_interest"]
text_clusters = pd.read_csv(PROCESSED / "text_clusters.csv", index_col="code")["hdbscan"]
titles = pd.read_csv(PROCESSED / "occupations.csv", index_col="code")["title"]

def field_purity(labels: pd.Series) -> float:
    """Average share of a Phase 2 text cluster that lands in a single group (higher = fields kept together)."""
    tc = text_clusters.loc[labels.index]
    keep = tc != -1
    ct = pd.crosstab(tc[keep], labels[keep])
    return (ct.max(axis=1) / ct.sum(axis=1)).mean()

## 1. Sweep the text weight and number of groups
- **RIASEC ARI**: agreement with each job's top career interest (higher = better fit for "I want to…" browsing)
- **Field purity**: how well jobs from the same field stay together

In [2]:
rows = []
for w in [0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 1.0]:
    H = hybrid_matrix(w)
    for k in [6, 8]:
        l = pd.Series(KMeans(k, n_init=10, random_state=42).fit_predict(H), index=H.index)
        rows.append({"text_weight": w, "groups": k,
                     "RIASEC_ARI": round(adjusted_rand_score(interests.loc[l.index], l), 3),
                     "field_purity": round(field_purity(l), 2),
                     "smallest": np.bincount(l).min(), "largest": np.bincount(l).max()})
sweep = pd.DataFrame(rows)
sweep

,text_weight,groups,RIASEC_ARI,field_purity,smallest,largest
0,0.0,6,0.237,0.64,121,201
1,0.0,8,0.217,0.60,83,155
2,0.3,6,0.199,0.69,123,185
3,0.3,8,0.249,0.71,75,157
4,0.5,6,0.354,0.75,76,249
5,0.5,8,0.239,0.74,69,140
6,0.6,6,0.351,0.78,75,247
7,0.6,8,0.232,0.76,70,155
8,0.7,6,0.348,0.80,72,250
9,0.7,8,0.231,0.77,70,158


**Result:** a mix beats either source alone on interest agreement, while keeping most of the field purity of
text-only groups. We use **text weight = 0.7, 6 groups**. (Weight 0 = numeric only, as in Phase 3; weight 1 = text only.)

## 2. Do the problem jobs land in sensible places now?

In [3]:
numeric = pd.read_csv(PROCESSED / "numeric_clusters.csv", index_col="code")["kmeans6"]
groups = hybrid_groups()
probe = ["Registered Nurses", "Nursing Assistants", "Security Managers", "Facilities Managers",
         "Preschool Teachers, Except Special Education", "Chief Executives", "Software Developers"]

def mates(labels, code, n=4):
    same = labels.index[(labels == labels[code]) & (labels.index != code)]
    return "; ".join(titles.loc[same].sample(n, random_state=0))

rows = []
for p in probe:
    c = titles.index[titles == p][0]
    rows.append({"job": p, "numeric-only group (sample)": mates(numeric, c), "hybrid group (sample)": mates(groups, c)})
pd.set_option("display.max_colwidth", 110)
pd.DataFrame(rows)

,job,numeric-only group (sample),hybrid group (sample)
0,Registered Nurses,"Ambulance Drivers and Attendants, Except Emergency Medical Technicians; First-Line Supervisors of Mechanic...",Paramedics; Ophthalmic Medical Technologists; Marriage and Family Therapists; Medical and Clinical Laborat...
1,Nursing Assistants,"Postal Service Mail Sorters, Processors, and Processing Machine Operators; Tailors, Dressmakers, and Custo...",Emergency Medical Technicians; Neurodiagnostic Technologists; Marriage and Family Therapists; Orthoptists
2,Security Managers,"Ambulance Drivers and Attendants, Except Emergency Medical Technicians; First-Line Supervisors of Mechanic...",Appraisers of Personal and Business Property; Loan Interviewers and Clerks; Credit Counselors; Appraisers ...
3,Facilities Managers,"Ambulance Drivers and Attendants, Except Emergency Medical Technicians; First-Line Supervisors of Mechanic...",Appraisers of Personal and Business Property; Loan Interviewers and Clerks; Credit Counselors; Appraisers ...
4,"Preschool Teachers, Except Special Education","Bill and Account Collectors; File Clerks; Appraisers and Assessors of Real Estate; Securities, Commodities...","Anthropology and Archeology Teachers, Postsecondary; Area, Ethnic, and Cultural Studies Teachers, Postseco..."
5,Chief Executives,Political Scientists; Instructional Coordinators; Family Medicine Physicians; Financial Managers,Appraisers of Personal and Business Property; Loan Interviewers and Clerks; Credit Counselors; Appraisers ...
6,Software Developers,Web Developers; Environmental Compliance Inspectors; Transportation Engineers; Environmental Economists,Biochemists and Biophysicists; Nuclear Power Reactor Operators; Bioinformatics Technicians; Mechatronics E...


## 3. The final hierarchy

In [4]:
tree = json.loads((ROOT / "web" / "data" / "hierarchy.json").read_text(encoding="utf-8"))
count = lambda x: 1 if "code" in x else sum(count(c) for c in x["children"])
rows = []
for g in tree["children"]:
    subs = g["children"]
    rows.append({"group": g["name"], "tagline": g["tagline"], "jobs": count(g),
                 "subclusters": sum(s["name"] != "Unique roles" for s in subs),
                 "unique roles": sum(count(s) for s in subs if s["name"] == "Unique roles"),
                 "examples": " | ".join(s["name"] for s in subs if s["name"] != "Unique roles")[:150]})
pd.DataFrame(rows)

,group,tagline,jobs,subclusters,unique roles,examples
0,Business & Management,"I want to run things: manage, sell, organize and handle money",176,11,13,Values · Land · Owners | Law Enforcement · Criminal · Suspects | Motions · Trial · Judicial | Securities ·...
1,Build & Fix,"I want to build, repair and operate machines and equipment",251,16,25,Needles · Fabric · Sew | Decks · Crew · Marine | Drive · Equipment · Trucks | Sharpen · Cuts · Workpieces ...
2,Teach & Guide,"I want to teach, train and guide people",72,2,0,Findings · Funding · Events | School · Meetings · Training
3,Help & Heal,I want to help people and care for their health,123,4,0,Dental · Teeth · Jaw | Visual · Eye · Lenses | Samples · Cell · Laboratory Equipment | Physical · Care · P...
4,Tech & Analysis,"I want to solve problems with technology, science and data",195,15,13,Editing · Film · Studio | Design Ideas · Sketches · Furniture | Projects · Cost · Managers Plan | Vulnerab...
5,Hands-On Service,"I want active, hands-on work serving people",106,7,15,Foods · Cooking · Dishes | Traffic · Passenger · Delays | Animal · Treatment · Bathe | Rules · Housekeepin...


## 4. Summary
- **Level 1:** 6 hybrid groups (70% text, 30% numeric), named by their defining attributes.
- **Level 2:** HDBSCAN subclusters inside each group, labeled with TF-IDF against their sibling clusters.
- **Level 3:** subclusters over 30 jobs are split again with K-Means (~15 jobs each).
- Jobs HDBSCAN can't place go into a **"Unique roles"** bubble in their group, matching the README's goal of
  keeping niche roles separate instead of forcing them into a cluster.

Rebuild with: `.venv\\Scripts\\python src\\build_hierarchy.py`